# Storing data in Databases: Local MySQL and MySQL on AWS RDS and Google Cloud SQL - 01/27/2026

In [1]:
!pip install mysql-connector-python -q
!pip install google-cloud-storage -q
!pip install pymysql -q
!pip install sqlalchemy -q

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'c:\\programdata\\anaconda3\\envs\\ds\\lib\\site-packages\\google\\auth\\app_engine.py'
Consider using the `--user` option or check the permissions.



## Required Packages

Before running this notebook, make sure you have the following packages installed:

| Package | Purpose |
|---------|---------|
| `mysql-connector-python` | Official MySQL driver for Python, provides connectivity to MySQL databases |
| `pymysql` | Pure-Python MySQL client library, alternative driver used by SQLAlchemy |
| `sqlalchemy` | SQL toolkit and ORM for Python, provides a high-level interface to databases |
| `google-cloud-storage` | Google Cloud Storage client library (for GCP integration examples) |
| `pandas` | Data manipulation and analysis (usually pre-installed in most environments) |

The cell below will install these packages automatically using `pip`.


In [3]:
import pandas as pd
import mysql.connector as mysql

In [ ]:
# Make sure the cell above runs without errors

# 1. Access local MySQL database from Python

In the context of databases, a **cursor** is a database object used to retrieve rows from a result set one at a time. You can think of a cursor as a pointer to one row in a set of rows. The set of rows the cursor holds is called the active set.

In the Python code below, `connection.cursor()` creates a new cursor object using the active database connection. You can then call `cursor.execute(sql_query)` to execute a SQL query against the database. The `cursor.fetchall()` method retrieves all the rows of a query result, returning them as a list of tuples. Each tuple corresponds to a row in the result. In the code below:

1. `pymysql.connect()` establishes a connection to the MySQL database.
2. `connection.cursor()` creates a new cursor object. In the context of databases, a cursor is a database object that retrieves data from a result set one row at a time. 
3. `cursor.execute("SELECT count(*) from customer")` executes a SQL query that counts all the rows in the "customer" table.
4. `result = cursor.fetchall()` fetches all the rows from the last executed SQL query.
5. The `for` loop prints each row in the result.
6. Finally, `connection.close()` closes the connection to the database. It's important to always close the connection when you're done with it to free up resources.

In Python, the `with` statement is used for exception handling and it simplifies the management of resources like file streams or database connections. This concept is known as context management.

When you use a `with` statement to manage the cursor, Python automatically calls the `__enter__` method to set up the resource (in this case, the cursor), and then it calls the `__exit__` method to clean up the resource when the `with` block is exited, even if an exception occurred within the block.

In the context of your code:

```python
with connection.cursor() as cursor:
    # Execute a query
    cursor.execute("SELECT count(*) from customer")
    # Fetch all rows from the last executed query
    result = cursor.fetchall()
```
Note: 

The `with` statement is used to create a new cursor from the database connection. When the `with` block is exited (either normally after the `fetchall` method is called, or if an exception occurs within the block), Python automatically closes the cursor. This is efficient and safe, because it ensures that the cursor is always closed properly, even if an error occurs. It's a good practice to use the `with` statement for resource management in Python.

### Method 1: hard-code the connection parameters

In [27]:
## Note: First, you need to create and populate the database
### To create the database, run the olist-db-creation-script.sql
### To populate the database, run the populate_tables_command-line_solution.sql

import pymysql

# Connect to the database. Change and use your own credentials.
connection = pymysql.connect(host='localhost', user='root', password='root', database='olist_db')

try:
    with connection.cursor() as cursor:
        # Execute a query
        cursor.execute("SELECT count(*) from customer")
        
        # Fetch all rows from the last executed query
        result = cursor.fetchall()
        
        for row in result:
            print(row)
finally:
    connection.close()


(99441,)


### Method 2: Retrieve the connection parameteters from db_config.ini

In [5]:
pwd

'c:\\Users\\aacastellanosb\\Dropbox\\William & Mary\\Teaching\\Spring 2026\\Week 1 - Introduction'

In [19]:
import pymysql
import configparser

# Read database configuration from a file
config = configparser.ConfigParser()
config.read(r"C:\\Users\\aacastellanosb\\Dropbox\William & Mary\\Teaching\\Spring 2026\\Week 1 - Introduction\\Olist-db-lab\\db_config.ini")

# Connect to the database
connection = pymysql.connect(host='localhost', 
                             user=config.get('mysql', 'user'), 
                             password=config.get('mysql', 'password'), 
                             database=config.get('mysql', 'database'))

try:
    with connection.cursor() as cursor:
        # Execute a query
        cursor.execute("SELECT count(*) from customer")
        
        # Fetch all rows from the last executed query
        result = cursor.fetchall()
        
        for row in result:
            print(row)
finally:
    connection.close()

(99441,)


# 2. Access AWS Relational Database System (RDS)

This Python script is used to connect to a MySQL database, execute a SQL query, and load the results into a Pandas DataFrame.

1. `import pandas as pd` and `import pymysql`: These lines import the required libraries. Pandas is a data analysis library and PyMySQL is a MySQL client for Python.

2. The next few lines define the credentials and details required to connect to the MySQL database:
    - `hostname`: The host where the MySQL database is located.
    - `port`: The port number to connect to the MySQL server. The default MySQL port is 3306.
    - `dbname`: The name of the database to connect to.
    - `username` and `db_pwd`: The username and password to authenticate with the MySQL server.

3. `conn_string = f'mysql+pymysql://{username}:{db_pwd}@{hostname}:{port}/{dbname}'`: This line creates the connection string that will be used to connect to the MySQL database. It uses the PyMySQL driver (`mysql+pymysql://`).

4. `sql_query = 'SELECT * FROM customer LIMIT 5'`: This line defines the SQL query that will be executed. In this case, it selects the first 5 rows from the "customer" table.

5. `df = pd.read_sql(sql_query, conn_string)`: This line uses the `read_sql` function from Pandas to execute the SQL query and load the results into a DataFrame. The `read_sql` function takes two arguments: the SQL query and the connection string.

6. `df`: This line displays the DataFrame. In a Jupyter notebook, just writing the variable name will display its contents.

In [ ]:
# RUN THIS CELL **ONLY** after you create the olist_db database using the olist-db-creation-script.sql found in Blackboard.

import pandas as pd
import pymysql

# Database credentials. Change and get your own credentials.
hostname = 'bigdata-wm.c15zrkxzw3cf.us-east-1.rds.amazonaws.com'
port = 3306  # Default MySQL port; adjust if necessary
dbname = 'olist_db'
username = 'root'
db_pwd = 'TribeWM2026'

# Create database connection string
conn_string = f'mysql+pymysql://{username}:{db_pwd}@{hostname}:{port}/{dbname}'

# Use Pandas to query the database; example query shown
sql_query = 'SELECT * FROM customer LIMIT 5'

# Load query results into a Pandas DataFrame
df = pd.read_sql(sql_query, conn_string)

# Display the DataFrame
df

# 3. Access Google Cloud SQL

This Python script is used to connect to a Google Cloud SQL instance, execute a SQL query, and load the results into a Pandas DataFrame.

1. `import pandas as pd` and `import pymysql`: These lines import the required libraries. Pandas is a data analysis library and PyMySQL is a MySQL client for Python.

2. The next few lines define the credentials and details required to connect to the Google Cloud SQL instance:
    - `hostname`: The public IP address of your Google Cloud SQL instance.
    - `port`: The port number to connect to the MySQL server. The default MySQL port is 3306.
    - `dbname`: The name of the database to connect to.
    - `username` and `db_pwd`: The username and password to authenticate with the MySQL server.

3. `conn_string = f'mysql+pymysql://{username}:{db_pwd}@{hostname}:{port}/{dbname}'`: This line creates the connection string that will be used to connect to the MySQL database. It uses the PyMySQL driver (`mysql+pymysql://`). Note that it's highly recommended to use a more secure method to handle passwords and credentials, such as environment variables or secret management services.

4. `sql_query = 'SELECT * FROM customer LIMIT 5'`: This line defines the SQL query that will be executed. In this case, it selects the first 5 rows from the "customer" table.

5. `df = pd.read_sql(sql_query, conn_string)`: This line uses the `read_sql` function from Pandas to execute the SQL query and load the results into a DataFrame. The `read_sql` function takes two arguments: the SQL query and the connection string.

6. `df`: This line displays the DataFrame. In a Jupyter notebook, just writing the variable name will display its contents.

In [8]:
import pandas as pd
import pymysql
import ssl
from urllib.parse import quote_plus
from sqlalchemy import create_engine

# Google Cloud SQL credentials and connection information
hostname = '136.107.95.155'  # Use the public IP address of your Cloud SQL instance
port = 3306  # Default MySQL port; adjust if necessary
dbname = 'olist_db'
username = 'root'
db_pwd = 'Tribe@WM2026'

# Create database connection string
# Note: It's highly recommended to use a more secure method to handle passwords and credentials,
# such as environment variables or secret management services.
# URL-encode the password to escape special characters like @
conn_string = f'mysql+pymysql://{username}:{quote_plus(db_pwd)}@{hostname}:{port}/{dbname}'

# Create SSL context (required for Cloud SQL remote connections)
ssl_context = ssl.create_default_context()
ssl_context.check_hostname = False
ssl_context.verify_mode = ssl.CERT_NONE

engine = create_engine(conn_string, connect_args={'ssl': ssl_context})
# Use Pandas to query the database; example query shown
sql_query = 'SELECT * FROM customer LIMIT 5'

# Load query results into a Pandas DataFrame
df = pd.read_sql(sql_query, engine)

# Display the DataFrame
df

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,00012a2ce6f8dcda20d059ce98491703,248ffe10d632bebe4f7267f1f44844c9,6273,osasco,SP
1,000161a058600d5901f007fab4c27140,b0015e09bb4b6e47c52844fab5fb6638,35550,itapecerica,MG
2,0001fd6190edaaf884bcaf3d49edf079,94b11d37cd61cb2994a194d11f89682b,29830,nova venecia,ES
3,0002414f95344307404f0ace7a26f1d5,4893ad4ea28b2c5b3ddf4e82e79db9e6,39664,mendonca,MG
4,000379cdec625522490c315e70c7a9fb,0b83f73b19c2019e182fd552c048a22c,4841,sao paulo,SP


# 4. Denormalizing the Data

The following SQL query is adapted to include all orders from the `olist_db` database, regardless of whether there is associated information in the linked tables. It uses `LEFT JOIN` to ensure that every order is returned, and it will show `NULL` for any missing data in the resulting output.

```sql
USE olist_db;
SELECT
    o.order_id,
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    oi.order_item_id,
    oi.product_id,
    oi.seller_id,
    oi.shipping_limit_date,
    oi.price,
    oi.freight_value,
    p.payment_sequential,
    p.payment_type,
    p.payment_installments,
    p.payment_value,
    r.review_id,
    r.review_score,
    pr.product_photos_qty,
    pr.product_length_cm,
    pr.product_width_cm,
    pc.product_category_name_english,
    c.customer_unique_id,
    c.customer_city,
    c.customer_state,
    s.seller_zip_code_prefix,
    s.seller_city,
    s.seller_state
FROM
    orders o
LEFT JOIN
    o_item oi ON o.order_id = oi.order_id
LEFT JOIN
    o_payment p ON o.order_id = p.order_id
LEFT JOIN
    review r ON o.order_id = r.order_id
LEFT JOIN
    product pr ON oi.product_id = pr.product_id
LEFT JOIN
    prod_cat_name_tr pc ON pr.product_category_name = pc.product_category_name
LEFT JOIN
    customer c ON o.customer_id = c.customer_id
LEFT JOIN
    seller s ON oi.seller_id = s.seller_id


In [9]:
import pandas as pd
import pymysql
import ssl
from urllib.parse import quote_plus
from sqlalchemy import create_engine

# Google Cloud SQL credentials and connection information
hostname = '136.107.95.155'  # Use the public IP address of your Cloud SQL instance
port = 3306  # Default MySQL port; adjust if necessary
dbname = 'olist_db'
username = 'root'
db_pwd = 'Tribe@WM2026'

# Create database connection string
conn_string = f'mysql+pymysql://{username}:{quote_plus(db_pwd)}@{hostname}:{port}/{dbname}'

# Create SSL context (required for Cloud SQL remote connections)
ssl_context = ssl.create_default_context()
ssl_context.check_hostname = False
ssl_context.verify_mode = ssl.CERT_NONE

engine = create_engine(conn_string, connect_args={'ssl': ssl_context})
# Use Pandas to query the database; example query shown


# Use Pandas to query the database; corrected query shown

'''
This query consolidates order, item, payment, review, product, category, customer, and seller information into one denormalized result set. It’s useful for detailed analysis of the complete order lifecycle—from purchase to delivery, including customer feedback and payment specifics.
'''

sql_query = """
SELECT
    o.order_id,
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    oi.order_item_id,
    oi.product_id,
    oi.seller_id,
    oi.shipping_limit_date,
    oi.price,
    oi.freight_value,
    p.payment_sequential,
    p.payment_type,
    p.payment_installments,
    p.payment_value,
    pr.product_photos_qty,
    pr.product_length_cm,

    pr.product_width_cm,
    pc.product_category_name_english,
    c.customer_unique_id,
    c.customer_city,
    c.customer_state,
    s.seller_zip_code_prefix,
    s.seller_city,
    s.seller_state
FROM
    orders o
LEFT JOIN
    o_item oi ON o.order_id = oi.order_id
LEFT JOIN
    o_payment p ON o.order_id = p.order_id
LEFT JOIN
    product pr ON oi.product_id = pr.product_id
JOIN
    prod_cat_name_tr pc ON pr.product_category_name = pc.product_category_name
LEFT JOIN
    customer c ON o.customer_id = c.customer_id
LEFT JOIN
    seller s ON oi.seller_id = s.seller_id
LIMIT 10000000
"""

# Load query results into a Pandas DataFrame
df = pd.read_sql(sql_query, engine)

# Display the DataFrame
df


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_item_id,product_id,seller_id,...,product_photos_qty,product_length_cm,product_width_cm,product_category_name_english,customer_unique_id,customer_city,customer_state,seller_zip_code_prefix,seller_city,seller_state
0,2f80a0b08926b808eafcaa9ceb2e7af4,01122215dd21ac872ae567ec4e351e01,delivered,2018-04-13 18:16:52,2018-04-14 02:13:28,2018-04-17 00:54:50,2018-04-17,1,018ca97302e4293050cc41413194bb19,6481e96574816ead57975da2c0f6d80d,...,266.0,1.0,28.0,agro_industry_and_commerce\r,4d285c90d3a4161d1bba6feed1072244,sao paulo,SP,18072,sorocaba,SP
1,ca00c2ba5781124bd9493423de6d8862,dafe240fa4132e5da366caf9f6f9caaf,delivered,2018-08-10 18:09:49,2018-08-10 18:25:13,2018-08-13 09:01:00,2018-08-17,1,026f43af35e7951067097527d5c31bcc,269cff2d3c8d205c11f37a52402ea93b,...,1075.0,1.0,52.0,agro_industry_and_commerce\r,a5e706a547ea42fde80e3cff5316c84b,juquitiba,SP,15803,catanduva,SP
2,ca00c2ba5781124bd9493423de6d8862,dafe240fa4132e5da366caf9f6f9caaf,delivered,2018-08-10 18:09:49,2018-08-10 18:25:13,2018-08-13 09:01:00,2018-08-17,2,026f43af35e7951067097527d5c31bcc,269cff2d3c8d205c11f37a52402ea93b,...,1075.0,1.0,52.0,agro_industry_and_commerce\r,a5e706a547ea42fde80e3cff5316c84b,juquitiba,SP,15803,catanduva,SP
3,21577126c19bf11a0b91592e5844ba78,1eebfdb7083031b40f727fb71f6cd5b2,delivered,2018-03-16 00:14:19,2018-03-16 17:28:45,2018-03-19 18:26:48,2018-03-29,1,07f01b6fcacc1b187a71e5074199db2d,6481e96574816ead57975da2c0f6d80d,...,430.0,1.0,63.0,agro_industry_and_commerce\r,25ba76039a1caff121dfcb0d66e54780,cabo frio,RJ,18072,sorocaba,SP
4,21577126c19bf11a0b91592e5844ba78,1eebfdb7083031b40f727fb71f6cd5b2,delivered,2018-03-16 00:14:19,2018-03-16 17:28:45,2018-03-19 18:26:48,2018-03-29,1,07f01b6fcacc1b187a71e5074199db2d,6481e96574816ead57975da2c0f6d80d,...,430.0,1.0,63.0,agro_industry_and_commerce\r,25ba76039a1caff121dfcb0d66e54780,cabo frio,RJ,18072,sorocaba,SP
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115876,e326a262b6e44dcbb1df6b74f9aea574,e35070616c6bd238074e2ee5131113fd,delivered,2017-10-24 12:49:41,2017-10-24 13:06:25,2017-10-25 19:57:40,2017-10-26,1,ffaaddefb271481c66d4bd79844ecdae,813348c996469b40f2e028d5429d3495,...,316.0,1.0,25.0,housewares\r,8158212aeee10a495fd38a7973b22b7d,sao caetano do sul,SP,13206,jundiai,SP
115877,11b33f091608b48c86184dcfd339dc9e,ad0cff75dce98e304964d36a660ca06d,delivered,2017-12-21 04:47:03,2017-12-21 04:59:32,2017-12-22 20:57:01,2017-12-23,2,ffbc83054b3741a8d67fc59d9cf9d42d,6edacfd9f9074789dad6d62ba7950b9c,...,231.0,3.0,16.0,housewares\r,25bd804c580bc1ee0c93d073fae09a84,sao paulo,SP,7135,guarulhos,SP
115878,8682a030870edcce6a8b8299b68aa4d6,626ee03ce70d55d537a1e5d9f2072261,delivered,2018-07-29 23:08:40,2018-07-31 04:31:27,2018-08-01 15:45:00,2018-08-02,1,ffc88104d219c1b767d566fd93653dd2,31be790e64fc99f8ff48ec2bd18a3104,...,428.0,2.0,33.0,housewares\r,5bcb3b3af07be38f0a4ed0433745481d,campinas,SP,9690,sao bernardo do campo,SP
115879,68005513d670f9fc899928aed4963d22,3d9511a1f386dd62e326b7bee5b1066e,delivered,2018-04-11 15:38:50,2018-04-11 15:51:22,2018-04-13 00:03:55,2018-04-20,1,ffe0fc4e02c3559643ac063fa5cf9d07,c68fb906c8f4b4b946d8386bfa6e5467,...,678.0,3.0,23.0,housewares\r,07c8fde388fee352835f27b3ef13ff12,sao paulo,SP,14870,jaboticabal,SP


## Save DataFrame to CSV

First, we need to save our DataFrame to a CSV file. This file will then be uploaded to Google Cloud Storage.


In [ ]:
# Save DataFrame to CSV
df.to_csv('output_query.csv', index=False)

# Assignment 1 Exercises: Olist SQL Queries

Below are five queries you can use to analyze the Olist dataset. Each query focuses on a different aspect of the e-commerce data—from geographic distribution to payment patterns.

Deliverable:

One PDF. with the filename in the following format:

wmuser_assignment1.pdf

For example, in my case: aacastellanos_assignment1.pdf

The file should only contain the problem number and the queries. No need to put your name or user in the document.

1. **Number of orders per state:** <br />
SELECT * <br />
FROM TABLE <br />
Rest of query <br /><br />
... <br />
... <br />
...

5.  **Average Delivery time by State:** <br />
SELECT * <br />
FROM TABLE <br />
Rest of query <br />

---

## 1. Number of Orders per State

**Question:** How many orders are there from each customer state?

```sql
--Query here


## 2. Top 5 States with the Highest Average Freight Cost

**Question:**: Which states have the highest average freight value for orders?

```sql
--query here


## 3. Top 5 Product Categories by Total Revenue 

**Question:** Which product categories generate the most revenue? Show the top 5 categories with their total sales (price + freight).

```sql
-- Query here


---

## 4. Payment Method Distribution

**Question:** What percentage of orders use each payment type (credit_card, boleto, voucher, debit_card)? Round to 2 decimal places.

```sql
-- Query here
```


---

## 5. Average Delivery Time by State

**Question:** What is the average number of days between order purchase and delivery for each customer state? Only consider delivered orders. Sort by average delivery time descending.

*Hint: Use `DATEDIFF()` to calculate the difference between `order_delivered_customer_date` and `order_purchase_timestamp`.*

```sql
-- Query here
```
